[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_17/NB_bishop_ch17_gans.ipynb)

# Chapter 17 -- Generative Adversarial Networks

This notebook covers:
- GAN on MNIST (generator + discriminator)
- Mode collapse demonstration
- Wasserstein GAN (WGAN-GP)
- Latent space interpolation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print(f'TF {tf.__version__}, GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 1. Data Preparation

In [ ]:
(X_train, y_train), (_, _) = keras.datasets.mnist.load_data()
X_train = X_train.astype(np.float32) / 255.0
X_train = X_train.reshape(-1, 28, 28, 1) * 2.0 - 1.0

BATCH_SIZE, NOISE_DIM = 256, 100
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(60000).batch(BATCH_SIZE)
print(f'Data: {X_train.shape}, range [{X_train.min():.0f}, {X_train.max():.0f}]')

## 2. Generator and Discriminator

In [ ]:
def build_generator(noise_dim=100):
    return keras.Sequential([
        layers.Dense(7*7*256, use_bias=False, input_shape=(noise_dim,)),
        layers.BatchNormalization(), layers.LeakyReLU(0.2), layers.Reshape((7, 7, 256)),
        layers.Conv2DTranspose(128, 5, 1, 'same', use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(64, 5, 2, 'same', use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(1, 5, 2, 'same', use_bias=False, activation='tanh')])

def build_discriminator():
    return keras.Sequential([
        layers.Conv2D(64, 5, 2, 'same', input_shape=(28, 28, 1)),
        layers.LeakyReLU(0.2), layers.Dropout(0.3),
        layers.Conv2D(128, 5, 2, 'same'),
        layers.LeakyReLU(0.2), layers.Dropout(0.3),
        layers.Flatten(), layers.Dense(1)])

generator = build_generator(NOISE_DIM)
discriminator = build_discriminator()
print(f'Generator params: {generator.count_params():,}')
print(f'Discriminator params: {discriminator.count_params():,}')

In [ ]:
generator.summary()

In [ ]:
discriminator.summary()

## 3. Training Loop

In [ ]:
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)

def disc_loss(real_out, fake_out):
    return cross_entropy(tf.ones_like(real_out), real_out) + cross_entropy(tf.zeros_like(fake_out), fake_out)

def gen_loss(fake_out):
    return cross_entropy(tf.ones_like(fake_out), fake_out)

gen_opt = keras.optimizers.Adam(1e-4)
disc_opt = keras.optimizers.Adam(1e-4)
seed = tf.random.normal([16, NOISE_DIM])

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], NOISE_DIM])
    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        gen_imgs = generator(noise, training=True)
        real_out = discriminator(images, training=True)
        fake_out = discriminator(gen_imgs, training=True)
        gl = gen_loss(fake_out)
        dl = disc_loss(real_out, fake_out)
    gen_opt.apply_gradients(zip(gt.gradient(gl, generator.trainable_variables), generator.trainable_variables))
    disc_opt.apply_gradients(zip(dt.gradient(dl, discriminator.trainable_variables), discriminator.trainable_variables))
    return gl, dl

In [ ]:
EPOCHS = 50
g_losses, d_losses = [], []
epoch_images = {}

for epoch in range(EPOCHS):
    start = time.time()
    eg, ed = [], []
    for batch in dataset:
        gl, dl = train_step(batch)
        eg.append(gl.numpy()); ed.append(dl.numpy())
    g_losses.append(np.mean(eg)); d_losses.append(np.mean(ed))
    if (epoch+1) in [1, 5, 10, 20, 30, 50]:
        epoch_images[epoch+1] = generator(seed, training=False).numpy()
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} -- G:{g_losses[-1]:.4f} D:{d_losses[-1]:.4f} -- {time.time()-start:.1f}s')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, EPOCHS+1), g_losses, label='Generator', color='#1a3a6e', lw=1.5)
ax.plot(range(1, EPOCHS+1), d_losses, label='Discriminator', color='#cd0000', lw=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('GAN Training Losses', fontweight='bold')
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch17_gan_training')
plt.show()

## 4. Generated Images Over Epochs

In [ ]:
show_epochs = sorted(epoch_images.keys())
fig, axes = plt.subplots(len(show_epochs), 8, figsize=(16, 2.5*len(show_epochs)))
for row, ep in enumerate(show_epochs):
    imgs = epoch_images[ep]
    for col in range(8):
        axes[row, col].imshow((imgs[col,:,:,0]+1)/2, cmap='gray'); axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'Ep {ep}', fontsize=12, rotation=0, labelpad=40)
fig.suptitle('GAN Generated Images Over Training', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch17_generated')
plt.show()

In [ ]:
# Final 64 samples
final = generator(tf.random.normal([64, NOISE_DIM]), training=False).numpy()
fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow((final[i,:,:,0]+1)/2, cmap='gray'); ax.axis('off')
fig.suptitle('GAN -- 64 Generated Digits', fontsize=14)
fig.tight_layout()
plt.show()

## 5. Latent Space Interpolation

In [ ]:
z1 = tf.random.normal([1, NOISE_DIM])
z2 = tf.random.normal([1, NOISE_DIM])
alphas = np.linspace(0, 1, 10)
interp_z = np.array([z1.numpy()[0]*(1-a) + z2.numpy()[0]*a for a in alphas])
interp_imgs = generator(interp_z.astype(np.float32), training=False).numpy()

fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow((interp_imgs[i,:,:,0]+1)/2, cmap='gray'); ax.axis('off')
    ax.set_title(f'{alphas[i]:.1f}')
fig.suptitle('Latent Space Interpolation', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch17_interpolation')
plt.show()

## 6. Mode Collapse Experiment

In [ ]:
gen_mc = build_generator(NOISE_DIM)
disc_mc = build_discriminator()
gen_opt_mc = keras.optimizers.Adam(1e-5)
disc_opt_mc = keras.optimizers.Adam(1e-3)
mc_seed = tf.random.normal([16, NOISE_DIM])
mc_images = {}

for epoch in range(30):
    for batch in dataset:
        for _ in range(5):
            noise = tf.random.normal([tf.shape(batch)[0], NOISE_DIM])
            with tf.GradientTape() as tape:
                gi = gen_mc(noise, training=True)
                dl = disc_loss(disc_mc(batch, training=True), disc_mc(gi, training=True))
            disc_opt_mc.apply_gradients(zip(tape.gradient(dl, disc_mc.trainable_variables), disc_mc.trainable_variables))
        noise = tf.random.normal([tf.shape(batch)[0], NOISE_DIM])
        with tf.GradientTape() as tape:
            gi = gen_mc(noise, training=True)
            gl = gen_loss(disc_mc(gi, training=True))
        gen_opt_mc.apply_gradients(zip(tape.gradient(gl, gen_mc.trainable_variables), gen_mc.trainable_variables))
        break
    if (epoch+1) in [1, 5, 10, 20, 30]:
        mc_images[epoch+1] = gen_mc(mc_seed, training=False).numpy()
    if (epoch+1) % 10 == 0:
        print(f'Mode collapse -- Epoch {epoch+1}')

In [ ]:
mc_eps = sorted(mc_images.keys())
fig, axes = plt.subplots(len(mc_eps), 8, figsize=(16, 2.5*len(mc_eps)))
for row, ep in enumerate(mc_eps):
    for col in range(8):
        axes[row, col].imshow((mc_images[ep][col,:,:,0]+1)/2, cmap='gray'); axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'Ep {ep}', fontsize=12, rotation=0, labelpad=40)
fig.suptitle('Mode Collapse -- Unbalanced Training', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch17_mode_collapse')
plt.show()

## 7. Wasserstein GAN (WGAN-GP)

In [ ]:
gen_w = build_generator(NOISE_DIM)
disc_w = build_discriminator()
gen_opt_w = keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
disc_opt_w = keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
LAMBDA_GP = 10

def gradient_penalty(disc, real, fake):
    eps = tf.random.uniform([tf.shape(real)[0], 1, 1, 1], 0.0, 1.0)
    interp = eps * real + (1 - eps) * fake
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = disc(interp, training=True)
    grads = tape.gradient(pred, interp)
    return tf.reduce_mean((tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1,2,3]) + 1e-8) - 1.0)**2)

In [ ]:
@tf.function
def wgan_step(images):
    bs = tf.shape(images)[0]
    for _ in range(5):
        noise = tf.random.normal([bs, NOISE_DIM])
        with tf.GradientTape() as tape:
            fake = gen_w(noise, training=True)
            dl = tf.reduce_mean(disc_w(fake, training=True)) - tf.reduce_mean(disc_w(images, training=True)) + LAMBDA_GP * gradient_penalty(disc_w, images, fake)
        disc_opt_w.apply_gradients(zip(tape.gradient(dl, disc_w.trainable_variables), disc_w.trainable_variables))
    noise = tf.random.normal([bs, NOISE_DIM])
    with tf.GradientTape() as tape:
        fake = gen_w(noise, training=True)
        gl = -tf.reduce_mean(disc_w(fake, training=True))
    gen_opt_w.apply_gradients(zip(tape.gradient(gl, gen_w.trainable_variables), gen_w.trainable_variables))
    return gl, dl

In [ ]:
WE = 30
wg, wd = [], []
for epoch in range(WE):
    start = time.time()
    eg, ed = [], []
    for batch in dataset:
        gl, dl = wgan_step(batch)
        eg.append(gl.numpy()); ed.append(dl.numpy())
    wg.append(np.mean(eg)); wd.append(np.mean(ed))
    if (epoch+1) % 10 == 0:
        print(f'WGAN Epoch {epoch+1}/{WE} -- G:{wg[-1]:.4f} D:{wd[-1]:.4f} -- {time.time()-start:.1f}s')

In [ ]:
wgan_imgs = gen_w(tf.random.normal([64, NOISE_DIM]), training=False).numpy()
fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow((wgan_imgs[i,:,:,0]+1)/2, cmap='gray'); ax.axis('off')
fig.suptitle('WGAN-GP -- 64 Generated Digits', fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(g_losses, label='Generator', color='#1a3a6e')
axes[0].plot(d_losses, label='Discriminator', color='#cd0000')
axes[0].set_title('Vanilla GAN', fontweight='bold'); axes[0].legend()
axes[1].plot(wg, label='Generator', color='#1a3a6e')
axes[1].plot(wd, label='Critic', color='#cd0000')
axes[1].set_title('WGAN-GP', fontweight='bold'); axes[1].legend()
fig.suptitle('Training Stability Comparison', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch17_wgan_comparison')
plt.show()

## Summary

- **Vanilla GAN**: generator vs discriminator adversarial training
- **Mode collapse**: generator produces limited variety
- **WGAN-GP**: Wasserstein distance + gradient penalty for stability
- **Latent interpolation** reveals smooth learned representations

In [ ]:
takeaways = [
    '1. GANs learn through adversarial training: generator vs discriminator.',
    '2. Mode collapse occurs when the generator produces limited variety.',
    '3. WGAN-GP uses Wasserstein distance and gradient penalty for stability.',
    '4. Balancing G/D learning rates is crucial.',
    '5. Latent space interpolation reveals smooth learned representations.'
]
print('Key Takeaways:')
for t in takeaways:
    print(t)